<a href="https://colab.research.google.com/github/ajaykumar080286/MachineLearning/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [87]:
#!pip install optuna
import pandas as pd
import numpy as np

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [65]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [66]:
df=pd.read_csv(url,names=columns)

In [67]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [68]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)
df.fillna(df.mean(), inplace=True)

In [69]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63,0
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27,0
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30,0
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47,1


In [70]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.00000,180.000000,32.9,0.171,63,0
764,2,122.0,70.0,27.00000,155.548223,36.8,0.340,27,0
765,5,121.0,72.0,23.00000,112.000000,26.2,0.245,30,0
766,1,126.0,60.0,29.15342,155.548223,30.1,0.349,47,1


In [71]:
X=df.drop('Outcome',axis=1)
y=df['Outcome']

In [72]:
X_train, X_test, y_train, y_test= train_test_split(X,y ,test_size=0.2, random_state=42)

In [73]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [74]:
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (614, 8)
Test set shape: (154, 8)


In [83]:
# Define the objective function
def objectiveFunction(trial):
  # Suggest values for the hyperparameters
  n_estimators=trial.suggest_int('n_estimators',50,200)
  max_depth=trial.suggest_int('max_depth',3,20)

   # Create the RandomForestClassifier with suggested hyperparameters

  model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,random_state=42)

  # Perform 3-fold cross-validation and calculate accuracy

  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score  # Return the accuracy score for Optuna to maximize

In [85]:
# Create a study object and optimize the objective function
study=optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objectiveFunction,n_trials=50)

[I 2025-12-19 12:33:25,213] A new study created in memory with name: no-name-469d916a-8bd3-45d4-b406-7f7a354a6c45
[I 2025-12-19 12:33:27,012] Trial 0 finished with value: 0.7784791965566714 and parameters: {'n_estimators': 174, 'max_depth': 5}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:27,686] Trial 1 finished with value: 0.7670891120675912 and parameters: {'n_estimators': 70, 'max_depth': 5}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:29,071] Trial 2 finished with value: 0.7540490993145226 and parameters: {'n_estimators': 164, 'max_depth': 3}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:29,680] Trial 3 finished with value: 0.755706998246453 and parameters: {'n_estimators': 61, 'max_depth': 9}. Best is trial 0 with value: 0.7784791965566714.
[I 2025-12-19 12:33:31,104] Trial 4 finished with value: 0.780113183484776 and parameters: {'n_estimators': 140, 'max_depth': 14}. Best is trial 4 with value: 0.780113183484

In [86]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7866411605292524
Best hyperparameters: {'n_estimators': 118, 'max_depth': 13}


In [90]:
# Train a RandomForestClassifier using the best hyperparameters from Optuna

best_model=RandomForestClassifier(**study.best_trial.params,random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


**Samplers in Optuna**

In [91]:
# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [93]:
study1 = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study1.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-12-19 13:06:17,810] A new study created in memory with name: no-name-3fd74018-131c-4eae-8e56-0660b549a8f1
[I 2025-12-19 13:06:20,712] Trial 0 finished with value: 0.75242308305436 and parameters: {'n_estimators': 122, 'max_depth': 3}. Best is trial 0 with value: 0.75242308305436.
[I 2025-12-19 13:06:23,622] Trial 1 finished with value: 0.7817551410808226 and parameters: {'n_estimators': 127, 'max_depth': 14}. Best is trial 1 with value: 0.7817551410808226.
[I 2025-12-19 13:06:25,539] Trial 2 finished with value: 0.7768770923003347 and parameters: {'n_estimators': 63, 'max_depth': 13}. Best is trial 1 with value: 0.7817551410808226.
[I 2025-12-19 13:06:27,592] Trial 3 finished with value: 0.7719910728519049 and parameters: {'n_estimators': 100, 'max_depth': 11}. Best is trial 1 with value: 0.7817551410808226.
[I 2025-12-19 13:06:31,847] Trial 4 finished with value: 0.7719432488442531 and parameters: {'n_estimators': 162, 'max_depth': 6}. Best is trial 1 with value: 0.78175514108

In [94]:

# Print the best result
print(f'Best trial accuracy: {study1.best_trial.value}')
print(f'Best hyperparameters: {study1.best_trial.params}')

Best trial accuracy: 0.7866411605292524
Best hyperparameters: {'n_estimators': 120, 'max_depth': 13}


In [95]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study1.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


**Grid Sampler**

In [96]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [97]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-12-19 13:10:27,952] A new study created in memory with name: no-name-32658a7c-eaf9-49d3-971d-e95636468986
[I 2025-12-19 13:10:29,360] Trial 0 finished with value: 0.7654391838036028 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7654391838036028.
[I 2025-12-19 13:10:32,039] Trial 1 finished with value: 0.7735772357723577 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 1 with value: 0.7735772357723577.
[I 2025-12-19 13:10:32,579] Trial 2 finished with value: 0.7687151283277539 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 1 with value: 0.7735772357723577.
[I 2025-12-19 13:10:33,645] Trial 3 finished with value: 0.7752351347042882 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 3 with value: 0.7752351347042882.
[I 2025-12-19 13:10:34,692] Trial 4 finished with value: 0.7703491152558585 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 3 with value: 0.775235

In [98]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Best trial accuracy: 0.7817391997449387
Best hyperparameters: {'n_estimators': 50, 'max_depth': 10}
Test Accuracy with best hyperparameters: 0.75


**Optuna Visualizations**

In [99]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [100]:
# 1. Optimization History
plot_optimization_history(study).show()

In [101]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [102]:
# 3. Slice Plot
plot_slice(study).show()

In [103]:
# 4. Contour Plot
plot_contour(study).show()

In [104]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

**Optimizing Multiple ML Models**

In [105]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [106]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [107]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-12-19 13:27:39,387] A new study created in memory with name: no-name-2f634b9c-94a5-453c-a915-cf8fe6439704
[I 2025-12-19 13:27:44,409] Trial 0 finished with value: 0.752415112386418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 232, 'learning_rate': 0.01209759020070973, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.752415112386418.
[I 2025-12-19 13:27:49,960] Trial 1 finished with value: 0.7687151283277539 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 265, 'learning_rate': 0.020266419166077427, 'max_depth': 8, 'min_samples_split': 8, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.7687151283277539.
[I 2025-12-19 13:27:50,787] Trial 2 finished with value: 0.7703172325840906 and parameters: {'classifier': 'RandomForest', 'n_estimators': 175, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 2 with value: 0.7703172325840906.
[I 2025-12-1

In [108]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'RandomForest', 'n_estimators': 246, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 1, 'bootstrap': False}
Best trial accuracy: 0.7817391997449387


In [109]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.752415,2025-12-19 13:27:39.391774,2025-12-19 13:27:44.409398,0 days 00:00:05.017624,NaN,NaN,GradientBoosting,NaN,NaN,0.012098,13.0,8.0,10.0,232.0,COMPLETE
1,1,0.768715,2025-12-19 13:27:44.414816,2025-12-19 13:27:49.960395,0 days 00:00:05.545579,NaN,NaN,GradientBoosting,NaN,NaN,0.020266,8.0,3.0,8.0,265.0,COMPLETE
2,2,0.770317,2025-12-19 13:27:49.961328,2025-12-19 13:27:50.786962,0 days 00:00:00.825634,NaN,False,RandomForest,NaN,NaN,NaN,10.0,8.0,6.0,175.0,COMPLETE
3,3,0.778463,2025-12-19 13:27:50.788000,2025-12-19 13:27:52.100050,0 days 00:00:01.312050,NaN,False,RandomForest,NaN,NaN,NaN,17.0,2.0,8.0,252.0,COMPLETE
4,4,0.773593,2025-12-19 13:27:52.101031,2025-12-19 13:27:53.309928,0 days 00:00:01.208897,NaN,False,RandomForest,NaN,NaN,NaN,13.0,3.0,5.0,233.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.770325,2025-12-19 13:30:27.935000,2025-12-19 13:30:29.369530,0 days 00:00:01.434530,NaN,False,RandomForest,NaN,NaN,NaN,20.0,2.0,7.0,276.0,COMPLETE
96,96,0.770317,2025-12-19 13:30:29.370493,2025-12-19 13:30:30.557227,0 days 00:00:01.186734,NaN,False,RandomForest,NaN,NaN,NaN,19.0,5.0,6.0,241.0,COMPLETE
97,97,0.775211,2025-12-19 13:30:30.558303,2025-12-19 13:30:31.913175,0 days 00:00:01.354872,NaN,False,RandomForest,NaN,NaN,NaN,18.0,2.0,8.0,257.0,COMPLETE
98,98,0.684051,2025-12-19 13:30:31.914284,2025-12-19 13:30:31.953564,0 days 00:00:00.039280,13.529949,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [110]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
RandomForest,79
GradientBoosting,11
SVM,10


In [111]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.754810
RandomForest,0.773570
SVM,0.735493


In [112]:
# 1. Optimization History
plot_optimization_history(study).show()

In [113]:
# 3. Slice Plot
plot_slice(study).show()

In [114]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [116]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 5.2 MB/s eta 0:00:00


In [117]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()

[W 2025-12-19 13:32:12,604] Study instance does not contain trials.
